# Stage 14 V1 — TabPFN-3 large-context против GBDT_mean

## Experiment Lock

Data_final.xlsb SHA-256 fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930; working 289614; target DefMark; INN не predictor; 47 accepted features без Q_B1_norm/Q_B2_norm; outer CV 3 folds, shuffle, seed 42; fold seeds 43/44/45; saved GBDT_mean Gini 0.8063993952. tabpfn 8.5.0, source 9ed44abd5882140b88c9f2816c5791987ce059b9, locked TabPFN-3 checkpoint и H100 80 GB. Final test не используется.

Canonical flow: flags → guards/data → Stage14LiveStatus → run_smoke_preflight → run_full_context_preflight → compute gate → run_full_oof. Live-status является только слоем наблюдаемости и не меняет protocol.


### ОПЕРАТОР

RUN_SMOKE_PREFLIGHT=True — только smoke preflight. RUN_FULL_CONTEXT_PREFLIGHT=True — дорогой full-context preflight. RUN_FULL_OOF=True — полный 3-fold OOF; автоматически не включать. В сохранённом default state все три flags False.


In [ ]:
from __future__ import annotations
import dataclasses,hashlib,importlib.metadata,json,os,platform,subprocess,threading,time
from contextlib import contextmanager
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score
from sklearn.model_selection import StratifiedKFold
# ОПЕРАТОР: изменяйте только нужный флаг после review.
RUN_SMOKE_PREFLIGHT=False
RUN_FULL_CONTEXT_PREFLIGHT=False
RUN_FULL_OOF=False
ROOT=Path.cwd() if (Path.cwd()/'reports').is_dir() else Path.cwd().parent;GEN=ROOT/'reports'/'generated';DATA=ROOT/'data'/'raw'/'Data_final.xlsb'
S1=GEN/'stage1_baseline_results_V2.json';S7=GEN/'stage7_tabm_stacking_results_V1.json';OOF=GEN/'stage7_tabm_stacking_oof_V1.npz'
DATA_SHA='fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930';WI_SHA='80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45';CKPT_SHA='1f9949fd52e20b6ed8ece20b3b37242d2abbbf337d14b461a4244f9318031a88'
REPO='Prior-Labs/tabpfn_3';REV='1419f233c94070103b26cd156362bec024764be1';NAME='tabpfn-v3-classifier-v3_20260417_binary.ckpt';N=289614;SEEDS=(43,44,45);CAP=32768;TOL=1e-5;LIMIT_HOURS=24.;SAFETY=1.5
os.environ['TABPFN_MAX_BATCHED_TEST_ROWS']=str(CAP)
LOCK={'n_estimators':8,'categorical_features_indices':None,'softmax_temperature':.9,'balance_probabilities':False,'average_before_softmax':False,'device':'cuda:0','ignore_pretraining_limits':False,'inference_precision':'autocast','fit_mode':'fit_with_cache','memory_saving_mode':True,'keep_cache_on_device':True,'kv_cache_precision':'int8','n_preprocessing_jobs':1,'differentiable_input':False,'eval_metric':None,'tuning_config':None,'show_progress_bar':False}


### Guards и data preparation

Проверяются immutable dataset, feature, comparator и fold contracts до любого model call. Materialize только canonical working rows; final-test rows не передаются модели.


In [ ]:
def h(p):
 d=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):d.update(b)
 return d.hexdigest()
def safe(x):
 if dataclasses.is_dataclass(x):return safe(dataclasses.asdict(x))
 if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)):return [safe(v) for v in x]
 if isinstance(x,np.generic):return x.item()
 if isinstance(x,Path):return str(x)
 return x
def metric(y,p):
 z=p>=.5;a=float(roc_auc_score(y,p));return {'Gini':2*a-1,'ROC-AUC':a,'PR-AUC':float(average_precision_score(y,p)),'Precision@0.5':float(precision_score(y,z,zero_division=0)),'Recall@0.5':float(recall_score(y,z,zero_division=0)),'F1@0.5':float(f1_score(y,z,zero_division=0))}
assert all(p.exists() for p in (DATA,S1,S7,OOF)) and h(DATA)==DATA_SHA
s1=json.loads(S1.read_text(encoding='utf8'));s7=json.loads(S7.read_text(encoding='utf8'));features=list(s1['допустимые_признаки'])
assert len(features)==47 and features==s7['raw_features_in_order'] and not {'Q_B1_norm','Q_B2_norm','INN'}&set(features)
with np.load(OOF,allow_pickle=False) as z:wi=np.asarray(z['working_indices'],dtype=np.int64);y=np.asarray(z['target'],dtype=np.int8);fold=np.asarray(z['fold'],dtype=np.int8);base=np.asarray(z['gbdt_mean'],dtype=float)
assert len(wi)==N and hashlib.sha256(wi.tobytes()).hexdigest()==WI_SHA and abs(metric(y,base)['Gini']-.8063993952)<=1e-9
raw=pd.read_excel(DATA,engine='pyxlsb');X=raw.loc[wi,features].to_numpy(dtype=np.float32,copy=True);assert X.shape==(N,47) and np.array_equal(raw.loc[wi,'DefMark'].to_numpy(dtype=np.int8),y)
splits=list(StratifiedKFold(3,shuffle=True,random_state=42).split(np.zeros(N),y));expected=np.zeros(N,dtype=np.int8)
for i,(_,v) in enumerate(splits,1):expected[v]=i
assert np.array_equal(fold,expected);FINAL_TEST_USED=False;assert FINAL_TEST_USED is False
print('PASS: контракт 47 признаков, сохранённый GBDT_mean и final-test guard проверены.')


## Stage14LiveStatus

### Что проверяется и зачем

Panel обновляется примерно каждые 7 секунд в current cell. Background heartbeat читает только фактические CUDA/RAM counters, не вызывает model API и не подавляет исходные exceptions. Новая session сбрасывает handle и state.


In [ ]:
class Stage14LiveStatus:
 def __init__(self,heartbeat=7.):self.heartbeat=heartbeat;self.lock=threading.RLock();self.handle=None;self.can_display=None;self.started=None;self.last=0.;self.state={}
 def start(self,stage,**kw):
  with self.lock:self.handle=None;self.can_display=None;self.started=time.monotonic();self.last=0.;self.state={'Этап':stage,'Статус':'RUNNING',**kw}
  self.render(True)
 def snap(self):
  s=dict(self.state);s['Прошло']=time.monotonic()-self.started
  try:
   import torch
   if torch.cuda.is_available():s.update({'GPU':torch.cuda.get_device_name(0),'VRAM allocated':int(torch.cuda.memory_allocated(0)),'VRAM reserved':int(torch.cuda.memory_reserved(0)),'VRAM peak':int(torch.cuda.max_memory_allocated(0))})
  except Exception:pass
  try:s['Host RAM']=int(psutil.Process().memory_info().rss)
  except Exception:pass
  return s
 def render(self,force=False):
  if not force and time.monotonic()-self.last<self.heartbeat:return
  self.last=time.monotonic();s=self.snap();lines=['Stage 14 V1']
  for k in ('Этап','Fold','Подэтап','Checkpoint','Context rows','Query rows','Chunk','Processed rows','Total rows','Статус'): 
   if k in s:lines.append('{}: {}'.format(k,s[k]))
  if s.get('Total rows'):lines.append('Прогресс: {:.2f}%'.format(100*s.get('Processed rows',0)/s['Total rows']))
  if s.get('Rows/sec'):
   lines.append('Строк/сек: {:.3f}'.format(s['Rows/sec']));remain=s.get('Total rows',0)-s.get('Processed rows',0)
   if remain>0:lines.append('ETA: {:.1f} сек'.format(remain/s['Rows/sec']))
  lines.append('Прошло: {:.1f} сек'.format(s['Прошло']))
  for k in ('GPU','VRAM allocated','VRAM reserved','VRAM peak','Host RAM'):
   if k in s:lines.append('{}: {}'.format(k,s[k]))
  text='\n'.join(lines)
  if self.can_display is not False:
   try:
    from IPython.display import Markdown,display
    view=Markdown('```text\n'+text+'\n```')
    if self.handle is None:self.handle=display(view,display_id=True)
    else:self.handle.update(view)
    self.can_display=True;return
   except Exception:self.can_display=False
  print(text,flush=True)
 def set(self,**kw):
  with self.lock:self.state.update(kw)
  self.render(kw.get('Статус') in ('PASS','WARN','STOP','ERROR'))
 @contextmanager
 def blocking(self,stage,**kw):
  self.start(stage,**kw);stop=threading.Event()
  def beat():
   while not stop.wait(self.heartbeat):self.render()
  th=threading.Thread(target=beat,daemon=True);th.start()
  try:yield self
  except BaseException:self.set(Статус='ERROR');raise
  else:self.set(Статус='PASS')
  finally:stop.set();th.join()
live=Stage14LiveStatus()
def demo_live_status(seconds=15,raise_error=False):
 # Synthetic test only: модель TabPFN не загружается.
 with live.blocking('Synthetic UX demo',Подэтап='background heartbeat',**{'Processed rows':0,'Total rows':100}):
  for i in range(seconds):time.sleep(1);live.set(**{'Processed rows':(i+1)*100//seconds})
  if raise_error:raise RuntimeError('Synthetic exception: исходный traceback не подавляется.')


## Canonical execution functions

### Что проверяется и зачем

Это единственные ML entrypoints. Live-status оборачивает model load, fit_with_cache, оба prediction calls, chunks, metrics и evidence save, но не меняет inputs, context, count predictions, metrics или decision logic.


In [ ]:
def runtime():
 import torch
 r={'python':platform.python_version(),'tabpfn':importlib.metadata.version('tabpfn'),'torch':torch.__version__,'cuda':torch.version.cuda,'available':torch.cuda.is_available(),'count':torch.cuda.device_count(),'nvidia_smi':None,'driver_version':None}
 try:
  probe=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True,check=False)
  if probe.returncode==0 and probe.stdout.strip():
   r['nvidia_smi']=probe.stdout.strip();r['driver_version']=probe.stdout.strip().split(',')[-1].strip()
 except FileNotFoundError:
  pass
 if r['available']:
  p=torch.cuda.get_device_properties(0);r['gpu']={'name':torch.cuda.get_device_name(0),'bytes':int(p.total_memory)}
 return r
def guard(r):
 if r['python']!='3.12.2':raise RuntimeError('STOP: требуется Python ровно 3.12.2.')
 if r['tabpfn']!='8.5.0' or not str(r['torch']).startswith('2.8.0') or not r['cuda'] or r['count']!=1:raise RuntimeError('STOP: tabpfn 8.5.0, CUDA torch 2.8.0 и один GPU обязательны; CPU fallback запрещён.')
 if not r['available'] or not r.get('driver_version'):raise RuntimeError('STOP: не удалось получить NVIDIA driver через nvidia-smi. GPU environment provenance не подтверждён.')
 if 'H100' not in r['gpu']['name'].upper() or r['gpu']['bytes']<79*1024**3:raise RuntimeError('STOP: требуется NVIDIA H100 80 GB.')
def construct(seed,stage,fold=None):
 from huggingface_hub import hf_hub_download
 from tabpfn import TabPFNClassifier
 with live.blocking(stage,Fold=fold,Подэтап='model load'):
  p=Path(hf_hub_download(repo_id=REPO,filename=NAME,revision=REV));assert h(p)==CKPT_SHA;c=TabPFNClassifier(model_path=p,random_state=seed,inference_config={'SUBSAMPLE_SAMPLES':None},**LOCK)
 live.set(Checkpoint=p.name);return c,p
def resolved(c):
 x=safe(c.inference_config_)
 if x.get('SUBSAMPLE_SAMPLES','missing') is not None:raise RuntimeError('HYPOTHESIS_INVALID_CONTEXT')
 return x
def effective(c,n):
 m=getattr(getattr(c,'executor_',None),'ensemble_members',None);rows=[] if m is None else [len(q.X_train) for q in m if hasattr(q,'X_train')]
 if not rows or any(q!=n for q in rows):raise RuntimeError('HYPOTHESIS_INVALID_CONTEXT')
 return rows
def fit(c,xt,yt,stage,fold):
 with live.blocking(stage,Fold=fold,Подэтап='fit_with_cache',**{'Context rows':len(xt)}):c.fit(xt,yt)
 return effective(c,len(xt)),resolved(c)
def predict(c,x,stage,fold,label):
 pieces=[];t=time.perf_counter();total=len(x);chunks=(total+CAP-1)//CAP
 for j,i in enumerate(range(0,total,CAP),1):
  with live.blocking(stage,Fold=fold,Подэтап=label,Chunk='{}/{}'.format(j,chunks),**{'Processed rows':i,'Total rows':total}):p=np.asarray(c.predict_proba(x[i:i+CAP]),dtype=float)
  if p.shape!=(min(CAP,total-i),2) or not np.isfinite(p).all() or not ((p>=0)&(p<=1)).all() or not np.allclose(p.sum(1),1,atol=1e-6):raise RuntimeError('STOP: invalid predict_proba.')
  pieces.append(p[:,1]);done=min(i+CAP,total);e=time.perf_counter()-t;live.set(**{'Processed rows':done,'Rows/sec':done/e if e else None})
 return np.concatenate(pieces),time.perf_counter()-t
def run_smoke_preflight():
 if not RUN_SMOKE_PREFLIGHT:raise RuntimeError('STOP: RUN_SMOKE_PREFLIGHT=False.')
 r=runtime();guard(r);tr,va=splits[0];c,p=construct(SEEDS[0],'Smoke preflight',1);rows,cfg=fit(c,X[tr[:4096]],y[tr[:4096]],'Smoke preflight',1);a,_=predict(c,X[va[:256]],'Smoke preflight',1,'prediction 1');b,_=predict(c,X[va[:256]],'Smoke preflight',1,'prediction 2')
 return {'environment':r,'checkpoint_sha256':h(p),'context_rows':4096,'query_rows':256,'resolved_inference_config':cfg,'effective_context_rows':rows,'repeat_max_abs_diff':float(np.max(abs(a-b))),'validation_labels_used':False}
def run_full_context_preflight():
 if not RUN_FULL_CONTEXT_PREFLIGHT:raise RuntimeError('STOP: RUN_FULL_CONTEXT_PREFLIGHT=False.')
 import torch
 try:
  r=runtime();guard(r);tr,va=splits[0];torch.cuda.reset_peak_memory_stats(0);ram=psutil.Process().memory_info().rss;c,p=construct(SEEDS[0],'Full-context preflight',1);t=time.perf_counter();rows,cfg=fit(c,X[tr],y[tr],'Full-context preflight',1);cache=time.perf_counter()-t;a,infer=predict(c,X[va[:4096]],'Full-context preflight',1,'prediction 1');b,_=predict(c,X[va[:4096]],'Full-context preflight',1,'prediction 2');d=float(np.max(abs(a-b)))
 except torch.OutOfMemoryError as ex:
  live.set(Статус='STOP',Outcome='STOPPED_BY_COMPUTE_COST',Причина='CUDA OOM на locked full-context configuration; качество TabPFN-3 остаётся UNKNOWN.')
  raise RuntimeError('STOPPED_BY_COMPUTE_COST: CUDA OOM на locked full-context configuration.') from ex
 if d>TOL:raise RuntimeError('STOP: repeatability > 1e-5.')
 return {'environment':r,'checkpoint_sha256':h(p),'context_rows':len(tr),'query_rows':4096,'resolved_inference_config':cfg,'effective_context_rows':rows,'context_invariant':'PASS','cache_build_seconds':cache,'inference_seconds':infer,'rows_per_second':4096/infer,'peak_vram_bytes':int(torch.cuda.max_memory_allocated(0)),'host_ram_rss_delta_bytes':int(psutil.Process().memory_info().rss-ram),'prediction_1':a,'prediction_2':b,'max_abs_diff':d,'repeatability_tolerance':TOL,'validation_labels_used':False}


## Compute gate и full OOF

### Что проверяется и зачем

Compute gate использует только full_context_evidence. Full OOF сохраняет partial evidence после завершённого fold без overwrite; при default False ничего не создаётся.


In [ ]:
def section(kind,lines):
 print('\n'.join(['Stage 14 V1 · {}'.format(kind),'']+['- '+x for x in lines]))
def smoke_summary(e):
 r=e['environment'];section('Smoke preflight',['environment/provenance: PASS','Python: {}'.format(r['python']),'TabPFN: {}'.format(r['tabpfn']),'Torch/CUDA: {} / {}'.format(r['torch'],r['cuda']),'GPU: {}'.format(r['gpu']['name']),'checkpoint SHA: PASS','probability checks: PASS','max_abs_diff: {:.3e}'.format(e['repeat_max_abs_diff']),'итог: PASS','','FACTS: smoke preflight завершён без использования validation labels и quality metrics.','INTERPRETATION: environment, checkpoint и повторяемость prediction готовы к следующему preflight.','LIMITATIONS: smoke использует context=4096 и query=256; full context не проверен.','NEXT STEP: при отдельном operator decision включить RUN_FULL_CONTEXT_PREFLIGHT=True.'])
def full_context_summary(e):
 ram=e.get('host_ram_rss_delta_bytes');ram_line='host RAM: недоступно' if ram is None else 'host RAM (RSS delta): {:.1f} MiB'.format(ram/1024**2);section('Full-context preflight',['Fold 1/3','expected context rows: {}'.format(e['context_rows']),'effective context rows: {}'.format(e['effective_context_rows']),'query rows = 4096','SUBSAMPLE_SAMPLES=None','context invariant: {}'.format(e['context_invariant']),'cache-build time: {:.1f} s'.format(e['cache_build_seconds']),'inference time: {:.1f} s'.format(e['inference_seconds']),'rows/sec: {:.1f}'.format(e['rows_per_second']),'max_abs_diff: {:.3e}'.format(e['max_abs_diff']),'tolerance <= 1e-5','peak VRAM: {:.1f} GiB'.format(e['peak_vram_bytes']/1024**3),ram_line,'итог: PASS','','FACTS: full context прошёл invariant и repeatability checks; quality metrics не рассчитывались.','INTERPRETATION: есть допустимое evidence для compute gate, но не для вывода о качестве TabPFN-3.','LIMITATIONS: выполнен только Fold 1/3 и query=4096; полный OOF не запускался.','NEXT STEP: review compute gate; RUN_FULL_OOF остаётся ручным флагом.'])
def compute_gate(e):
 total=SAFETY*(3*e['cache_build_seconds']+N/e['rows_per_second']);return {'projected_total_seconds':total,'projected_h100_hours':total/3600,'decision':'STOPPED_BY_COMPUTE_COST' if total>LIMIT_HOURS*3600 else 'ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW','run_full_oof_changed_automatically':False}
def compute_gate_summary(e,g):
 workload='3 × {:.1f} s cache-build + {:.0f} rows inference'.format(e['cache_build_seconds'],N);lines=['measured cache-build time: {:.1f} s'.format(e['cache_build_seconds']),'measured throughput: {:.1f} rows/sec'.format(e['rows_per_second']),'expected 3-fold workload: '+workload,'safety factor = 1.5','projected full OOF runtime: {:.3f} H100-hours'.format(g['projected_h100_hours']),'ceiling = 24 H100-hours','decision: {}'.format(g['decision'])]
 if g['decision']=='STOPPED_BY_COMPUTE_COST':lines+=['','Качество TabPFN-3 остаётся UNKNOWN.','Полный OOF запрещён текущим compute lock.','FACTS: measured full-context evidence превысило compute ceiling с safety factor.','INTERPRETATION: полный OOF не выполняется; quality comparison отсутствует.','LIMITATIONS: качество TabPFN-3 UNKNOWN.','NEXT STEP: не запускать RUN_FULL_OOF при текущем compute lock.']
 else:lines+=['','FACTS: measured full-context evidence находится в пределах ceiling.','INTERPRETATION: full OOF eligible только после явного ручного включения RUN_FULL_OOF=True.','LIMITATIONS: quality metrics всё ещё отсутствуют.','NEXT STEP: operator может отдельно включить RUN_FULL_OOF=True после review.']
 section('Compute gate',lines)
def save_partial(fid,valid,prob,row):
 path=GEN/'stage14_tabpfn3_partial_fold{}_V1.npz'.format(fid)
 with path.open('xb') as f:np.savez_compressed(f,fold=fid,indices=valid,probabilities=prob,runtime_seconds=row['runtime_seconds'],metrics=json.dumps(safe(row)),seed=row['seed'])
 return str(path.relative_to(ROOT))
def run_full_oof(gate):
 if not RUN_FULL_OOF:raise RuntimeError('STOP: RUN_FULL_OOF=False.')
 if gate['decision']!='ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW':raise RuntimeError('STOPPED_BY_COMPUTE_COST')
 r=runtime();guard(r);out=np.full(N,np.nan);rows=[];t=time.perf_counter()
 for fid,((tr,va),seed) in enumerate(zip(splits,SEEDS),1):
  fs=time.perf_counter();live.start('Full OOF',Fold='{}/3'.format(fid),Подэтап='подготовка fold');c,p=construct(seed,'Full OOF','{}/3'.format(fid));ctx,cfg=fit(c,X[tr],y[tr],'Full OOF','{}/3'.format(fid));prob,_=predict(c,X[va],'Full OOF','{}/3'.format(fid),'inference');out[va]=prob
  with live.blocking('Full OOF',Fold='{}/3'.format(fid),Подэтап='metrics'):tm,bm=metric(y[va],prob),metric(y[va],base[va])
  row={'fold':fid,'seed':seed,'context_rows':ctx,'resolved_inference_config':cfg,'metrics':tm,'GBDT_mean_metrics':bm,'delta_gini':tm['Gini']-bm['Gini'],'runtime_seconds':time.perf_counter()-fs}
  with live.blocking('Full OOF',Fold='{}/3'.format(fid),Подэтап='intermediate evidence save'):row['partial_evidence']=save_partial(fid,va,prob,row)
  rows.append(row);live.set(Статус='PASS',Подэтап='fold завершён: Gini {:.6f}'.format(tm['Gini']))
 tm,bm=metric(y,out),metric(y,base);d=tm['Gini']-bm['Gini'];ds=[q['delta_gini'] for q in rows];decision='material_gain' if d>=.01 and sum(q>0 for q in ds)>=2 else 'inferior' if d<=-.01 and sum(q<0 for q in ds)>=2 else 'no_material_benefit'
 return {'status':'completed','final_test_used':False,'fold_metrics':rows,'tabpfn_oof_metrics':tm,'gbdt_mean_oof_metrics':bm,'delta_gini':d,'decision':decision,'runtime_seconds':time.perf_counter()-t}
smoke_evidence=run_smoke_preflight() if RUN_SMOKE_PREFLIGHT else None
if smoke_evidence is not None:smoke_summary(smoke_evidence)
full_context_evidence=run_full_context_preflight() if RUN_FULL_CONTEXT_PREFLIGHT else None
if full_context_evidence is not None:full_context_summary(full_context_evidence)
gate=compute_gate(full_context_evidence) if full_context_evidence is not None else None
if gate is not None:compute_gate_summary(full_context_evidence,gate)
def full_oof_safety(e,g):
 if e is None:raise RuntimeError('STOP: сначала требуется успешный full-context preflight.')
 if e.get('context_outcome')=='HYPOTHESIS_INVALID_CONTEXT' or e.get('context_invariant')!='PASS':raise RuntimeError('STOP: full OOF запрещён: HYPOTHESIS_INVALID_CONTEXT.')
 if g is None:raise RuntimeError('STOP: сначала требуется compute gate.')
 if g.get('decision')=='STOPPED_BY_COMPUTE_COST':raise RuntimeError('STOP: полный OOF запрещён compute lock. Качество TabPFN-3 остаётся UNKNOWN.')
 if g.get('decision')!='ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW':raise RuntimeError('STOP: full OOF запрещён: gate не eligible.')
full_oof_result=None
if RUN_FULL_OOF:
 full_oof_safety(full_context_evidence,gate);full_oof_result=run_full_oof(gate)
print('FACTS: full OOF не выполнен; TabPFN quality metrics отсутствуют; final test не использован.' if full_oof_result is None else json.dumps(safe(full_oof_result),ensure_ascii=False,indent=2))
